In [34]:
import pandas as pd
import numpy as np
import matplotlib as plt
import pandas_datareader.data as web
from datetime import date
import statsmodels.api as sm

In [30]:
# Import relevant data
trading_1min = pd.read_csv("data/yfinance/1m_interval_trading_data.csv")
trading_2min = pd.read_csv("data/yfinance/2m_interval_trading_data.csv")
trading_1h = pd.read_csv("data/yfinance/1h_interval_trading_data.csv")

background_data = pd.read_csv("data/yfinance/ticker_background_data_df.csv")

# Daily Fama-French 5 Factors
start = "2020-01-01"
end = date.today()

ff5 = web.DataReader("F-F_Research_Data_5_Factors_2x3_daily", "famafrench", start, end)

# ff5 is a dict-like object; the actual data is in ff5[0]
factors = ff5[0]
print(factors.tail())

/var/folders/vl/1jjwbkdj1r126_dp4xttqycm0000gn/T/ipykernel_4578/49755974.py:12: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff5 = web.DataReader("F-F_Research_Data_5_Factors_2x3_daily", "famafrench", start, end)


            Mkt-RF   SMB   HML   RMW   CMA    RF
Date                                            
2026-06-24   -0.07  0.78 -0.21  0.42  0.61  0.01
2026-06-25   -0.13  0.61  0.91 -0.76  0.51  0.01
2026-06-26    0.15  1.36 -0.95  0.44  0.04  0.01
2026-06-29    1.20 -0.89 -0.90 -1.77 -0.50  0.01
2026-06-30    0.73 -0.10 -0.62 -1.10 -0.49  0.01


In [ ]:
dfs = {
    "1min": trading_1min,
    "2min": trading_2min,
    "1h": trading_1h,
}

for name, df in dfs.items():
    #Convert to EST
    df['Datetime'] = pd.to_datetime(df['Datetime'], utc=True).dt.tz_convert('America/New_York')
    df['Date'] = df['Datetime'].dt.normalize()

    df.sort_values(['Ticker', 'Datetime'], inplace=True)

    daily_close = df.groupby(['Ticker', 'Date'])['Close'].last().reset_index()
    daily_close = daily_close.sort_values(['Ticker', 'Date'])

    daily_close['Daily_Return'] = daily_close.groupby('Ticker')['Close'].pct_change()
    merged = df.merge(daily_close[['Ticker', 'Date', 'Daily_Return']], on=['Ticker', 'Date'], how='inner')

    # Both datasets are in EST, but strip it from base 
    merged['Date'] = merged['Date'].dt.tz_localize(None)
    merged = merged.merge(factors, left_on = 'Date', right_index=True, how='left')

    dfs[name] = merged  # store the result back in the dict

dfs['1h'][dfs['1h']['Ticker'] == 'AAPL'][['Date', 'Ticker', 'Daily_Return']].drop_duplicates().head(10)

/var/folders/vl/1jjwbkdj1r126_dp4xttqycm0000gn/T/ipykernel_4578/4000367083.py:16: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  daily_close['Daily_Return'] = daily_close.groupby('Ticker')['Close'].pct_change()


,Date,Ticker,Daily_Return
0,2024-08-02,AAPL,NaN
7,2024-08-05,AAPL,-0.048271
14,2024-08-06,AAPL,-0.009871
21,2024-08-07,AAPL,0.013205
28,2024-08-08,AAPL,0.016582
35,2024-08-09,AAPL,0.013875
42,2024-08-12,AAPL,0.005779
49,2024-08-13,AAPL,0.017100
56,2024-08-14,AAPL,0.001582
63,2024-08-15,AAPL,0.013943
